In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv(r"C:\Users\thana\Desktop\flight_analysis\data\flight_ml_ready.csv")

print("Γραμμές:", len(df))
print("Έτοιμο!")

Γραμμές: 1928371
Έτοιμο!


In [2]:
# Features βάσει παρατηρήσεων EDA
features = [
    "Month_sin", "Month_cos", # cyclical month
    "DayOfWeek", "IS_WEEKEND",  # Ημέρα
    "CRSDepTime",   # Ώρα αναχώρησης
    "Distance",
    "CarrierDelay", "NASDelay"  # delay causes (χωρίς leakage)
]

le = LabelEncoder()
df["Carrier_encoded"] = le.fit_transform(df["UniqueCarrier"])
features.append("Carrier_encoded")

X = df[features]
y = df["DELAYED"]

print("Features:", features)
print("X shape", X.shape)
print("y distribution:\n", y.value_counts())

Features: ['Month_sin', 'Month_cos', 'DayOfWeek', 'IS_WEEKEND', 'CRSDepTime', 'Distance', 'CarrierDelay', 'NASDelay', 'Carrier_encoded']
X shape (1928371, 9)
y distribution:
 DELAYED
0    1097891
1     830480
Name: count, dtype: int64


In [3]:
# Train/test split & scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", len(X_train))
print("Test size", len(X_test))

Train size: 1542696
Test size 385675


In [6]:
# Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
y_pred_nb = nb.predict(X_test_scaled)

print("=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))
print("ROC-AUC", roc_auc_score(y_test, nb.predict_proba(X_test_scaled)[:,1]))

=== Naive Bayes ===
              precision    recall  f1-score   support

           0       0.75      0.94      0.83    219579
           1       0.87      0.59      0.70    166096

    accuracy                           0.79    385675
   macro avg       0.81      0.76      0.77    385675
weighted avg       0.80      0.79      0.78    385675

ROC-AUC 0.8225721891416561


In [4]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1]))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.76      0.88      0.81    219579
           1       0.80      0.62      0.70    166096

    accuracy                           0.77    385675
   macro avg       0.78      0.75      0.76    385675
weighted avg       0.78      0.77      0.77    385675

ROC-AUC: 0.837224605514298


In [5]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:,1]))

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.81      0.86      0.84    219579
           1       0.81      0.74      0.77    166096

    accuracy                           0.81    385675
   macro avg       0.81      0.80      0.81    385675
weighted avg       0.81      0.81      0.81    385675

ROC-AUC: 0.8858060321796788


### Σύγκριση αποτελεσμάτων
|**Μοντέλο** |**Accuracy** | **F1(avg)** | **ROC_AOC**|
|:-----------|:-----------:|:-----------:|-----------:|
|Naive Bayes |79%          |0.77         |0.823       |
|Logistic Regression|77%   |0.76         |0.837       |
|Random Forest|81%         |0.81         |0.886       |